# Week 73, Day 1: Project layout

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import os, sys, importlib

os.makedirs("shop/payments", exist_ok=True)
open("shop/__init__.py", "w").write("print('loading shop')\nNAME = 'shop'\n")
open("shop/payments/__init__.py", "w").write("")
open("shop/payments/cards.py", "w").write("from .. import NAME\n\ndef charge(n):\n    return f'{NAME} charged {n}'\n")

sys.path.insert(0, os.getcwd())
from shop.payments.cards import charge
print(charge(5))
import shop
print(sorted(k for k in sys.modules if k.startswith("shop")))

In [ ]:
import os, sys

os.makedirs("textkit", exist_ok=True)
open("textkit/__init__.py", "w").write('''"""Small text utilities."""
from .clean import normalise
from .count import word_count

__all__ = ["normalise", "word_count"]
__version__ = "0.3.0"
''')
open("textkit/clean.py", "w").write('''def normalise(text):
    return " ".join(text.lower().split())
''')
open("textkit/count.py", "w").write('''from .clean import normalise

def word_count(text):
    return len(normalise(text).split()) if text.strip() else 0
''')

sys.path.insert(0, os.getcwd())
import textkit
print(textkit.word_count("  Hello   World  "), textkit.normalise("A  B"), textkit.__version__)
from textkit import *
print(sorted(n for n in dir() if n in ("normalise", "word_count", "clean")))

In [ ]:
import os, sys, runpy, importlib
from importlib.resources import files

os.makedirs("wordy/data", exist_ok=True)
open("wordy/__init__.py", "w").write("")
open("wordy/data/stopwords.txt", "w").write("the\na\nand\nof\n")
open("wordy/cli.py", "w").write('''import sys
from importlib.resources import files

def stopwords():
    return set(files("wordy").joinpath("data/stopwords.txt").read_text().split())

def main(argv=None):
    words = (argv if argv is not None else sys.argv[1:])
    kept = [w for w in words if w.lower() not in stopwords()]
    print(" ".join(kept))
    return 0
''')
open("wordy/__main__.py", "w").write('''import sys
from wordy.cli import main

sys.exit(main())
''')

sys.path.insert(0, os.getcwd())
sys.argv = ["wordy", "The", "joy", "of", "Python", "and", "tests"]
try:
    runpy.run_module("wordy", run_name="__main__")      # what python -m wordy does
except SystemExit as e:
    print("exit status", e.code)
print(sorted(files("wordy").joinpath("data/stopwords.txt").read_text().split()))

> **Key idea.** A package is a folder of modules with an __init__.py, imported under one dotted name; importing a submodule imports its parents first, once each. Relative imports (from . import x) work only inside packages; __all__ lists the public names. The src layout puts the package under src/, so tests import the installed package (pip install -e . for development), with tests in tests/ and settings in pyproject.toml. python -m package runs __main__.py, which should only call main(); console-script entry points make commands. importlib.resources reads a package's own data files. Files named like standard modules shadow them.

### Exercise: From paths to module names

Write `module_name(path)`: given a path inside a src layout, such as `"src/textkit/count.py"`, return the dotted module name `"textkit.count"`. An `__init__.py` gives the package's own name (`"src/textkit/payments/__init__.py"` gives `"textkit.payments"`); a `__main__.py` gives the package name plus `".__main__"`. Paths must start with `src/` and end in `.py`; anything else raises `ValueError`. Accept `\\` (Windows) as a separator too.

In [ ]:
def module_name(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"paths to dotted names\"\n    assert module_name(\"src/textkit/count.py\") == \"textkit.count\"\n    assert module_name(\"src/textkit/__init__.py\") == \"textkit\"\n    assert module_name(\"src/textkit/payments/__init__.py\") == \"textkit.payments\"\n    assert module_name(\"src/textkit/__main__.py\") == \"textkit.__main__\"\n    assert module_name(\"src\\\\textkit\\\\data\\\\load.py\") == \"textkit.data.load\"\n    for bad in [\"textkit/count.py\", \"src/textkit/notes.txt\", \"src/\"]:\n        try:\n            module_name(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Finding packages

Write `find_packages(paths)`, like the build tools' package discovery: `paths` is a list of file paths in a project (with `/`). Return the sorted dotted names of every **regular package** under `src/`: a folder that contains an `__init__.py` **and** whose parent folders up to `src/` are packages too (a folder with an `__init__.py` inside a folder without one isn't reachable as a subpackage, so it doesn't count).

In [ ]:
def find_packages(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_find():\n    \"package discovery\"\n    paths = [\n        \"pyproject.toml\", \"src/shop/__init__.py\", \"src/shop/cart.py\",\n        \"src/shop/payments/__init__.py\", \"src/shop/payments/cards.py\",\n        \"src/shop/data/prices.csv\", \"src/shop/data/extra/__init__.py\",\n        \"src/tools/__init__.py\", \"tests/__init__.py\", \"tests/test_cart.py\",\n    ]\n    assert find_packages(paths) == [\"shop\", \"shop.payments\", \"tools\"]\n    assert find_packages([\"src/a.py\", \"README.md\"]) == []")

### Exercise: A layout checker

Write `check_layout(paths)`: return a sorted list of problem messages for a project's file paths:
- `"no pyproject.toml"` if there's no `pyproject.toml` at the top;
- `"flat layout: NAME"` for each top-level folder (other than `src`, `tests`, `docs` and `scripts`) that has an `__init__.py` directly in it;
- `"tests inside the package: PATH"` for each file under `src/` whose file name starts with `test_`;
- `"shadows the standard library: PATH"` for each `.py` file whose name (without `.py`) is in `STDLIB`, anywhere in the project;
- `"no __init__.py: FOLDER"` for each folder under `src/` that contains `.py` files but no `__init__.py` (written as the folder path, such as `src/shop/utils`).

In [ ]:
STDLIB = {"random", "json", "statistics", "email", "typing", "test", "string", "csv", "math"}

def check_layout(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checker():\n    \"every kind of problem\"\n    good = [\"pyproject.toml\", \"README.md\", \"src/shop/__init__.py\", \"src/shop/cart.py\", \"tests/test_cart.py\"]\n    assert check_layout(good) == []\n    bad = [\"setup.py\", \"shop/__init__.py\", \"shop/cart.py\", \"src/tools/__init__.py\",\n           \"src/tools/test_tools.py\", \"src/tools/utils/helpers.py\", \"scripts/random.py\", \"src/tools/json.py\"]\n    assert check_layout(bad) == [\n        \"flat layout: shop\",\n        \"no __init__.py: src/tools/utils\",\n        \"no pyproject.toml\",\n        \"shadows the standard library: scripts/random.py\",\n        \"shadows the standard library: src/tools/json.py\",\n        \"tests inside the package: src/tools/test_tools.py\",\n    ]")

### Exercise: Make a package

Write a program that **creates** a package `greeter` in the current folder: `greeter/__init__.py` (re-exporting `greet` from `greeter/core.py`, with `__version__ = "1.0.0"` and `__all__ = ["greet"]`), `greeter/core.py` (with `greet(name, shout=False)` returning `"Hello, NAME!"`, upper-cased if `shout`), `greeter/cli.py` (with `main(argv)`: it prints the greeting for `argv[0]`, shouting if `"--shout"` is among the arguments, and returns `0`) and `greeter/__main__.py` (which calls `sys.exit(main(sys.argv[1:]))`). Use relative imports inside the package.

In [ ]:
import os

os.makedirs("greeter", exist_ok=True)

def write(path, text):
    with open(path, "w") as f:
        f.write(text)

write("greeter/__init__.py", ___)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_package():\n    \"an importable, runnable package\"\n    import importlib, sys, runpy, io, contextlib\n    for name in [m for m in sys.modules if m == \"greeter\" or m.startswith(\"greeter.\")]:\n        del sys.modules[name]\n    sys.path.insert(0, \".\")\n    try:\n        g = importlib.import_module(\"greeter\")\n        assert g.greet(\"Ada\") == \"Hello, Ada!\" and g.greet(\"Ada\", shout=True) == \"HELLO, ADA!\"\n        assert g.__version__ == \"1.0.0\" and g.__all__ == [\"greet\"]\n        assert \"from .\" in read_file(\"greeter/__init__.py\") and \"from .\" in read_file(\"greeter/cli.py\")\n        buf = io.StringIO()\n        old = sys.argv\n        sys.argv = [\"greeter\", \"Bob\", \"--shout\"]\n        try:\n            with contextlib.redirect_stdout(buf):\n                runpy.run_module(\"greeter\", run_name=\"__main__\")\n            assert False, \"__main__ should call sys.exit\"\n        except SystemExit as e:\n            assert e.code == 0\n        finally:\n            sys.argv = old\n        assert buf.getvalue().strip() == \"HELLO, BOB!\"\n    finally:\n        sys.path.remove(\".\")")

### Exercise: A package's own data

The page's program creates a package `quotes` with a data file `quotes/data/quotes.txt` (one quote per line). Write `quotes/__init__.py`'s function, `load()`: it returns the list of non-empty lines from the data file, read with **`importlib.resources.files`**, not with `open` and a relative path. Your program should write the file `quotes/__init__.py`; the checks import it from another folder, where a relative path would fail.

In [ ]:
import os

os.makedirs("quotes/data", exist_ok=True)
with open("quotes/data/quotes.txt", "w") as f:
    f.write("Beautiful is better than ugly.\n\nSimple is better than complex.\n")

with open("quotes/__init__.py", "w") as f:
    f.write('''
def load():
    with open("quotes/data/quotes.txt") as f:
        return [line for line in f.read().splitlines() if line]
''')

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_resources():\n    \"reads its own data wherever it's run from\"\n    import importlib, sys, os\n    sys.modules.pop(\"quotes\", None)\n    here = os.getcwd()\n    sys.path.insert(0, here)\n    os.makedirs(\"elsewhere\", exist_ok=True)\n    os.chdir(\"elsewhere\")\n    try:\n        q = importlib.import_module(\"quotes\")\n        assert q.load() == [\"Beautiful is better than ugly.\", \"Simple is better than complex.\"]\n    finally:\n        os.chdir(here)\n        sys.path.remove(here)\n    assert \"importlib.resources\" in read_file(\"quotes/__init__.py\")")

### Bug hunt: A helper that hid a library

The program saves its helper functions as `statistics.py`, so `import statistics` finds that file instead of the standard library, and `statistics.mean` doesn't exist. Rename the helper module (to `stats_helpers.py`) and import it under that name, so this prints `3 4.25`.

In [ ]:
import os, sys, importlib
sys.path.insert(0, os.getcwd())

with open("statistics.py", "w") as f:
    f.write("def spread(xs):\n    return max(xs) - min(xs)\n")
importlib.invalidate_caches()
sys.modules.pop("statistics", None)

import statistics
helpers = statistics

data = [2, 3, 5, 7]
print(helpers.spread([1, 4]), statistics.mean(data))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no shadowing\"\n    expect_output(\"3 4.25\")\n    assert not os.path.exists(\"statistics.py\") or \"spread\" not in read_file(\"statistics.py\")")

### Bug hunt: A package without an __init__.py

`calc/ops.py` uses `from .base import ZERO`, and `calc` has no `__init__.py`. That still imports here (as a namespace package), but `from calc import add` fails, because the package has no namespace of its own to re-export `add` from. Give `calc` an `__init__.py` that re-exports `add`, so this prints `5`.

In [ ]:
import os, sys, importlib
sys.path.insert(0, os.getcwd())
os.makedirs("calc", exist_ok=True)
with open("calc/base.py", "w") as f:
    f.write("ZERO = 0\n")
with open("calc/ops.py", "w") as f:
    f.write("from .base import ZERO\n\ndef add(a, b):\n    return ZERO + a + b\n")
importlib.invalidate_caches()

from calc import add
print(add(2, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a regular package\"\n    expect_output(\"5\")")

### Bug hunt: A relative import in a script

`tool.py` uses a relative import and is run directly as a script (here with `runpy.run_path`, as `python tool.py` would), so it fails with "attempted relative import with no known parent package". Make it a module of the package `kit` and run it **as a module** (`runpy.run_module("kit.tool", run_name="__main__")`, as `python -m kit.tool` would), so this prints `tool says: hi`.

In [ ]:
import os, sys, runpy, importlib
sys.path.insert(0, os.getcwd())
os.makedirs("kit", exist_ok=True)
with open("kit/__init__.py", "w") as f:
    f.write("")
with open("kit/words.py", "w") as f:
    f.write("WORD = 'hi'\n")
with open("kit/tool.py", "w") as f:
    f.write("from .words import WORD\nprint('tool says:', WORD)\n")
importlib.invalidate_caches()

runpy.run_path("kit/tool.py", run_name="__main__")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"run as part of its package\"\n    expect_output(\"tool says: hi\")")

### Puzzle: A tiny __main__.py

Arrange the lines of a package's `__main__.py` that hands everything to a tested `main` function and passes its exit status on. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the module name of src/shop/payments/__init__.py
q = ___   # how many times shop/__init__.py runs for "import shop.a" then "import shop.b"
r = ___   # True or False: with the src layout, tests can import the package without installing it, when run from the project folder
s = ___   # the file "python -m shop" runs, inside the shop package
t = ___   # True or False: "from . import x" works in a file run directly as a script

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"shop.payments\" and q == 1 and r is False, \"p, q and r\"\n    assert s == \"__main__.py\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an import graph

Write `import_graph(files)`: `files` maps paths in a src layout to their source code. For every module, parse its imports with `ast` and work out which **project** modules it imports (ignore anything that isn't one of the project's modules). Handle `import a.b`, `from a.b import c` (which imports `a.b.c` if that's a module, otherwise `a.b`), and relative imports (`from . import x`, `from .x import y`, `from ..x import y`, resolved against the module's package; in an `__init__.py` the package is the module itself). Importing `a.b` also imports its parent package `a`, so include parents that are project modules. Return a dict of module name to the sorted list of the project modules it imports (excluding itself). Then `cycles(graph)`: the sorted list of modules that are part of an import cycle.

In [ ]:
import ast

def import_graph(files):
    ___

def cycles(graph):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_graph():\n    \"imports between a project's modules\"\n    files = {\n        \"src/shop/__init__.py\": \"from .cart import Cart\\n\",\n        \"src/shop/cart.py\": \"import json\\nfrom shop.pricing import price\\nfrom . import util\\n\",\n        \"src/shop/pricing.py\": \"from .util import rounded\\nimport shop.payments.cards\\n\",\n        \"src/shop/util.py\": \"import math\\n\",\n        \"src/shop/payments/__init__.py\": \"\",\n        \"src/shop/payments/cards.py\": \"from ..cart import Cart\\nfrom .. import util\\n\",\n    }\n    g = import_graph(files)\n    assert g[\"shop\"] == [\"shop.cart\"]\n    assert g[\"shop.cart\"] == [\"shop\", \"shop.pricing\", \"shop.util\"]\n    assert g[\"shop.pricing\"] == [\"shop\", \"shop.payments\", \"shop.payments.cards\", \"shop.util\"]\n    assert g[\"shop.util\"] == [] and g[\"shop.payments\"] == []\n    assert g[\"shop.payments.cards\"] == [\"shop\", \"shop.cart\", \"shop.util\"]\n    assert cycles(g) == [\"shop\", \"shop.cart\", \"shop.payments.cards\", \"shop.pricing\"]\n    assert cycles({\"a\": [\"b\"], \"b\": []}) == [] and cycles({\"x\": [\"y\"], \"y\": [\"x\"], \"z\": [\"x\"]}) == [\"x\", \"y\"]")

## Warm-up

Importing `shop.payments.cards` imported **three** modules, outermost first: the package `shop` (running its `__init__.py`, once), the subpackage `shop.payments`, then `cards`. And `cards` reached back into its parent with a **relative** import, `from .. import NAME`. Packages are just folders of modules that Python treats as one dotted namespace.

## Packages

Week 20 Day 2 made modules: one `.py` file each. A **package** is a folder of modules imported under one name:

- A folder with an **`__init__.py`** file is a **regular package**. Importing the package runs its `__init__.py`, once (it's cached in `sys.modules`, Week 64 Day 5); whatever that file defines is the package's namespace. It's often empty, or it re-exports the public names: `from .cards import charge` lets users write `from shop import charge`.
- Folders inside are **subpackages**: `shop/payments/cards.py` is the module `shop.payments.cards`.
- A folder **without** `__init__.py` can still be imported, as a **namespace package**, which lets several separately installed distributions share one top-level name. For your own projects, always include `__init__.py`: forgetting it gives confusing results.
- Inside a package, **relative imports** name modules by their position: `from . import utils` (the same package), `from .utils import helper`, `from ..config import settings` (the parent package). They only work in modules imported **as part of a package**; running such a file directly as a script fails with `attempted relative import with no known parent package`.
- **`__all__`** in a module or `__init__.py` lists the names that `from package import *` exports, and documents the public interface. Names starting with `_` are private by convention.

## The src layout

A project is more than its package. The usual layout today, the **src layout**:

```text
textkit/                    # the project folder (a Git repository)
├── pyproject.toml          # the project's metadata and build settings (Day 2)
├── README.md
├── LICENSE
├── CHANGELOG.md            # Day 4
├── src/
│   └── textkit/            # the import package
│       ├── __init__.py
│       ├── __main__.py
│       ├── clean.py
│       ├── count.py
│       └── data/
│           └── stopwords.txt
└── tests/
    ├── conftest.py
    ├── test_clean.py
    └── test_count.py
```

Why put the package inside `src/`, rather than at the top (the **flat layout**)? When you run Python or pytest from the project folder, the current folder is on `sys.path`. With a flat layout, `import textkit` finds the **source folder** directly, so your tests pass even if the package is installed wrongly (a missing file, a forgotten data file): you're testing the folder, not what users get. With the src layout, `textkit` can only be imported once it's **installed**, normally as an **editable install** (`pip install -e .`), which links the environment to your source so edits take effect immediately. The tests then import the package the way users will.

Other conventions:

- The **distribution name** (what you `pip install`, such as `python-textkit`) and the **import name** (`textkit`) can differ, but life is easier when they match. Import names are lower case, with underscores if needed.
- **Tests live outside the package**, in `tests/`, so they aren't installed with it.
- Scripts and one-off tools go in a `scripts/` folder or become **entry points** (below), not loose files next to the package.
- Configuration for the tools (pytest, linters, type checkers) goes in `pyproject.toml` too, under `[tool.pytest.ini_options]` and similar tables.

## Running a package, and its data

**`python -m package`** runs a package's **`__main__.py`**: the place for a package's command-line interface. `python -m textkit count notes.txt` works wherever the package is installed, with no path to a script. Keep `__main__.py` tiny: it should only call a `main()` function defined in a normal module, so the logic can be imported and tested.

**Entry points** go further: `pyproject.toml` can declare **console scripts**, such as `textkit = "textkit.cli:main"`, and installing the package creates a small `textkit` command that calls that function (Day 2).

**Data files.** A package that needs a data file (a word list, a template, default settings) mustn't open it by a path relative to the current folder, which is wherever the user happens to be. Ask the package for its own files with **`importlib.resources`**: `files("textkit").joinpath("data/stopwords.txt").read_text()`. This works however the package was installed, even from a zip file. (`__file__`-based paths work for normal installs, but `importlib.resources` is the reliable way.)

## Names that collide

Python searches `sys.path` in order, and the **folder of the script you run comes first**. So a file of yours named like a standard library module (`random.py`, `json.py`, `statistics.py`, `email.py`, `test.py`) **shadows** the real one, for your program and for every library it imports that uses the real one. The symptoms are baffling: `AttributeError: module 'random' has no attribute 'randint'`, or a circular-import error deep inside a library you didn't write. When an import behaves strangely, print `module.__file__` to see which file was really loaded.

The same applies to packages: don't call your project's package `test`, `typing` or `requests`. And check a name is free on the **Python Package Index** (PyPI) before you settle on it for a distribution (Day 3).

## Practice

## On your own computer

Set up a real src-layout project and install it in editable mode (Day 2 explains the `pyproject.toml`; this minimal one is enough):

```bash
mkdir -p textkit/src/textkit textkit/tests && cd textkit
```

```bash
printf '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "textkit"\nversion = "0.1.0"\n' > pyproject.toml
```

```bash
echo 'def shout(s): return s.upper()' > src/textkit/__init__.py && python -m venv .venv && . .venv/bin/activate && pip install -e .
```

```bash
cd /tmp && python -c "import textkit, sys; print(textkit.shout('it works'), textkit.__file__)"
```

The import works from any folder, because the package is **installed**, and `__file__` shows it's still your source file, because the install is **editable**.

## Quiz

## Daily challenge

## Before you go

Take a project folder from an earlier week and sketch it in the src layout. Tomorrow: **`pyproject.toml`**, the one file that describes a project to every tool: its name, version, dependencies, entry points and build system.